# Emotion Injection Experiment

This notebook demonstrates the core innovation: **computational emotions** that directly modulate how a language model processes information.

Unlike prompting a model to "act happy" or "be frustrated", we inject emotion adapters that literally change the hidden states during the forward pass. The model *cannot* process information the same way when in different emotional states.

## What This Notebook Does

1. **Loads a base language model** (Qwen 2.5 0.5B for quick testing)
2. **Wraps it with emotion adapter layers** at key depths
3. **Demonstrates how different emotions produce different outputs** for the same prompt
4. **Trains the adapters** on emotional response examples
5. **Visualizes** how emotions modulate hidden states

In [ ]:
# Install dependencies (run once)
!pip install -q torch transformers accelerate einops

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple, Any
import math

# Check device
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA: {torch.cuda.get_device_name()}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple Silicon MPS")
else:
    device = "cpu"
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

## 1. Define Emotion State

We use a 4D continuous emotion space:
- **Valence** (-1 to 1): Negative ↔ Positive
- **Arousal** (0 to 1): Calm ↔ Excited
- **Curiosity** (0 to 1): Focused ↔ Exploratory
- **Confidence** (0 to 1): Uncertain ↔ Certain

This allows for nuanced emotional states like "slightly anxious and very curious".

In [ ]:
@dataclass
class EmotionState:
    """Continuous emotional state in 4D space."""
    valence: float = 0.0      # -1 to 1
    arousal: float = 0.5      # 0 to 1
    curiosity: float = 0.5    # 0 to 1
    confidence: float = 0.5   # 0 to 1
    
    def __post_init__(self):
        self.valence = max(-1.0, min(1.0, self.valence))
        self.arousal = max(0.0, min(1.0, self.arousal))
        self.curiosity = max(0.0, min(1.0, self.curiosity))
        self.confidence = max(0.0, min(1.0, self.confidence))
    
    def to_tensor(self, device="cpu"):
        return torch.tensor(
            [self.valence, self.arousal, self.curiosity, self.confidence],
            dtype=torch.float32,
            device=device
        )
    
    def __str__(self):
        parts = []
        if self.valence > 0.5: parts.append("pleased")
        elif self.valence < -0.5: parts.append("frustrated")
        if self.arousal > 0.7: parts.append("excited")
        elif self.arousal < 0.3: parts.append("calm")
        if self.curiosity > 0.7: parts.append("curious")
        if self.confidence > 0.7: parts.append("confident")
        elif self.confidence < 0.3: parts.append("uncertain")
        return ", ".join(parts) if parts else "neutral"

# Preset emotions
EMOTIONS = {
    "neutral": EmotionState(0.0, 0.5, 0.5, 0.5),
    "curious": EmotionState(0.3, 0.6, 0.9, 0.5),
    "excited": EmotionState(0.8, 0.9, 0.7, 0.7),
    "frustrated": EmotionState(-0.6, 0.7, 0.2, 0.4),
    "confident": EmotionState(0.2, 0.3, 0.4, 0.9),
    "anxious": EmotionState(-0.3, 0.8, 0.3, 0.2),
    "playful": EmotionState(0.7, 0.7, 0.8, 0.6),
    "skeptical": EmotionState(-0.2, 0.4, 0.6, 0.3),
}

print("Emotion presets:")
for name, emotion in EMOTIONS.items():
    print(f"  {name}: {emotion}")

## 2. Emotion Adapter Layer

The core innovation. This adapter modulates hidden states based on emotion:

```
h' = h * (1 + α * scale(emotion)) + bias(emotion)
```

Where:
- `h`: original hidden states
- `emotion`: 4D vector projected to hidden dimension
- `scale/bias`: learned projections
- `α`: modulation strength

In [ ]:
class EmotionAdapter(nn.Module):
    """Modulates hidden states based on emotional state."""
    
    def __init__(
        self,
        hidden_dim: int,
        emotion_dim: int = 4,
        modulation_strength: float = 0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.modulation_strength = modulation_strength
        
        intermediate = hidden_dim // 4
        
        # Project emotion to feature space
        self.emotion_proj = nn.Sequential(
            nn.Linear(emotion_dim, intermediate),
            nn.GELU(),
            nn.Linear(intermediate, intermediate),
            nn.GELU(),
        )
        
        # Generate scale (multiplicative)
        self.scale_proj = nn.Sequential(
            nn.Linear(intermediate, hidden_dim),
            nn.Tanh(),
        )
        
        # Generate bias (additive)
        self.bias_proj = nn.Sequential(
            nn.Linear(intermediate, hidden_dim),
            nn.Tanh(),
        )
        
        # Learnable gate
        self.gate = nn.Parameter(torch.ones(1) * 0.5)
        
        # Initialize small
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, hidden_states, emotion_state):
        """Apply emotional modulation."""
        if emotion_state.dim() == 1:
            emotion_state = emotion_state.unsqueeze(0)
        
        if emotion_state.size(0) == 1 and hidden_states.size(0) > 1:
            emotion_state = emotion_state.expand(hidden_states.size(0), -1)
        
        # Project emotion
        emotion_features = self.emotion_proj(emotion_state)
        scale = self.scale_proj(emotion_features).unsqueeze(1)
        bias = self.bias_proj(emotion_features).unsqueeze(1)
        
        # Apply modulation
        gate = torch.sigmoid(self.gate)
        modulated = hidden_states * (1.0 + gate * self.modulation_strength * scale)
        modulated = modulated + gate * self.modulation_strength * bias
        
        return modulated
    
    def get_modulation(self, emotion_state):
        """Get scale and bias for visualization."""
        if emotion_state.dim() == 1:
            emotion_state = emotion_state.unsqueeze(0)
        
        with torch.no_grad():
            features = self.emotion_proj(emotion_state)
            scale = self.scale_proj(features)
            bias = self.bias_proj(features)
            return scale.squeeze(), bias.squeeze()

# Test adapter
adapter = EmotionAdapter(hidden_dim=512)
test_hidden = torch.randn(2, 10, 512)  # batch=2, seq=10, hidden=512
test_emotion = EMOTIONS["curious"].to_tensor()
output = adapter(test_hidden, test_emotion)
print(f"Input shape: {test_hidden.shape}")
print(f"Output shape: {output.shape}")
print(f"Change magnitude: {(output - test_hidden).abs().mean():.4f}")

## 3. Load Base Model

We use Qwen 2.5 0.5B for quick experimentation. The architecture works with any transformer.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # Small model for testing
# MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Larger, better results

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True,
    device_map="auto" if device == "cuda" else None
)

if device != "cuda":
    base_model = base_model.to(device)

# Get model info
hidden_dim = base_model.config.hidden_size
num_layers = base_model.config.num_hidden_layers

print(f"\nModel loaded!")
print(f"  Hidden dimension: {hidden_dim}")
print(f"  Number of layers: {num_layers}")
print(f"  Total parameters: {sum(p.numel() for p in base_model.parameters()):,}")

## 4. Create Emotional Model Wrapper

We inject emotion adapters at key depths in the network:
- ~25%: Early reasoning affected
- ~50%: Mid-level processing changed
- ~75%: High-level concepts modulated
- ~90%: Output selection influenced

In [ ]:
class EmotionalModel(nn.Module):
    """Wraps a language model with emotion adapters."""
    
    def __init__(self, base_model, hidden_dim, num_layers, modulation_strength=0.15):
        super().__init__()
        self.base_model = base_model
        self.hidden_dim = hidden_dim
        
        # Freeze base model
        for param in base_model.parameters():
            param.requires_grad = False
        
        # Adapter injection points
        self.adapter_indices = [
            int(num_layers * 0.25),
            int(num_layers * 0.5),
            int(num_layers * 0.75),
            int(num_layers * 0.9),
        ]
        
        # Create adapters
        self.adapters = nn.ModuleDict({
            str(idx): EmotionAdapter(hidden_dim, modulation_strength=modulation_strength)
            for idx in self.adapter_indices
        })
        
        self.current_emotion = None
        self._hooks = []
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward hooks to inject adapters."""
        def make_hook(adapter_key):
            def hook(module, input, output):
                if self.current_emotion is None:
                    return output
                
                if isinstance(output, tuple):
                    hidden = output[0]
                    rest = output[1:]
                else:
                    hidden = output
                    rest = None
                
                emotion = self.current_emotion.to(hidden.device)
                modulated = self.adapters[adapter_key](hidden, emotion)
                
                if rest is not None:
                    return (modulated,) + rest
                return modulated
            return hook
        
        # Find and hook transformer layers
        layer_idx = 0
        for name, module in self.base_model.named_modules():
            if 'layers.' in name and name.count('.') == 1:
                if layer_idx in self.adapter_indices:
                    h = module.register_forward_hook(make_hook(str(layer_idx)))
                    self._hooks.append(h)
                layer_idx += 1
    
    def set_emotion(self, emotion: EmotionState):
        """Set current emotion for generation."""
        self.current_emotion = emotion.to_tensor(device)
    
    def clear_emotion(self):
        """Clear emotion (baseline generation)."""
        self.current_emotion = None
    
    def forward(self, **kwargs):
        return self.base_model(**kwargs)
    
    def generate(self, input_ids, **kwargs):
        return self.base_model.generate(input_ids, **kwargs)
    
    def trainable_params(self):
        return sum(p.numel() for p in self.adapters.parameters())

# Create emotional model
emotional_model = EmotionalModel(
    base_model, 
    hidden_dim, 
    num_layers,
    modulation_strength=0.15
)

print(f"Emotional model created!")
print(f"  Adapter injection points: {emotional_model.adapter_indices}")
print(f"  Trainable parameters: {emotional_model.trainable_params():,}")
print(f"  Parameter ratio: {emotional_model.trainable_params() / sum(p.numel() for p in base_model.parameters()) * 100:.2f}%")

## 5. Test Emotional Generation

Let's see how different emotions affect the model's outputs for the same prompt.

In [ ]:
def generate_with_emotion(prompt: str, emotion: EmotionState, max_tokens: int = 60):
    """Generate text with a specific emotion."""
    emotional_model.set_emotion(emotion)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = emotional_model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def generate_baseline(prompt: str, max_tokens: int = 60):
    """Generate without emotional modulation."""
    emotional_model.clear_emotion()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = emotional_model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
# Test prompt
prompt = "The user asked about their failing code. The AI responds:"

print("=" * 60)
print(f"PROMPT: {prompt}")
print("=" * 60)

# Baseline (no emotion)
print("\n[BASELINE - No Emotion]")
print(generate_baseline(prompt))

# Different emotions
for name in ["curious", "frustrated", "excited", "confident", "skeptical"]:
    emotion = EMOTIONS[name]
    print(f"\n[{name.upper()}] ({emotion})")
    print(generate_with_emotion(prompt, emotion))

In [ ]:
# Another test - response to exciting news
prompt = "Great news! The experiments succeeded. Response:"

print("=" * 60)
print(f"PROMPT: {prompt}")
print("=" * 60)

for name in ["neutral", "excited", "skeptical", "curious"]:
    emotion = EMOTIONS[name]
    print(f"\n[{name.upper()}]")
    print(generate_with_emotion(prompt, emotion))

## 6. Train Emotion Adapters

Now let's train the adapters to produce emotionally-appropriate responses. We use synthetic training data pairing emotions with expected responses.

In [ ]:
# Training data: (prompt, emotion_name, target_response)
TRAINING_DATA = [
    # Curious
    ("What do you think?", "curious", "Oh interesting! I'm wondering what happens if we push this further. What aspects are you most interested in?"),
    ("Here's my code", "curious", "Hmm, let me dig into this. I notice something intriguing here..."),
    ("Any thoughts?", "curious", "Fascinating! I'm curious about the edge cases. What happens when..."),
    
    # Excited
    ("I discovered something!", "excited", "Oh wow! That's amazing! Tell me more!"),
    ("The tests pass", "excited", "Yes! Fantastic! I knew we could do it!"),
    ("It works!", "excited", "Brilliant! This is exactly what we needed!"),
    
    # Frustrated
    ("It keeps failing", "frustrated", "Ugh, this is so annoying. Let me look at this again..."),
    ("Another bug", "frustrated", "This is getting tiresome. There has to be something we're missing."),
    ("Third attempt failed", "frustrated", "Seriously? This is frustrating. Let me think..."),
    
    # Confident
    ("What should we do?", "confident", "The best approach is definitely X. I've seen this work well."),
    ("Are you sure?", "confident", "Yes, I'm certain. The reasoning is clear."),
    ("What's the answer?", "confident", "It's definitely this. No question about it."),
    
    # Skeptical
    ("This is perfect!", "skeptical", "Hmm, I'm not so sure. Have you considered..."),
    ("Everyone agrees", "skeptical", "Well, popular opinion isn't always right. Let me push back."),
    ("It's the best way", "skeptical", "I have doubts. What about the edge cases?"),
    
    # Playful
    ("This is boring", "playful", "Ha! Let's spice things up then! What if we tried something wild?"),
    ("Routine task", "playful", "Ooh, let's make this fun! What if we added..."),
    
    # Anxious
    ("Critical system", "anxious", "Oh, let's be very careful here. I want to double-check everything."),
    ("Deadline soon", "anxious", "That's tight. We need to prioritize. What's essential?"),
]

print(f"Training examples: {len(TRAINING_DATA)}")

In [ ]:
# Training loop
from torch.optim import AdamW
from tqdm.auto import tqdm

# Only train adapter parameters
optimizer = AdamW(emotional_model.adapters.parameters(), lr=1e-4, weight_decay=0.01)

def train_step(prompt: str, emotion_name: str, target: str):
    """Single training step."""
    emotion = EMOTIONS[emotion_name]
    emotional_model.set_emotion(emotion)
    
    # Prepare input
    full_text = f"{prompt} {target}"
    inputs = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)
    
    # Prepare labels (only loss on target)
    prompt_len = len(tokenizer(prompt).input_ids)
    labels = inputs.input_ids.clone()
    labels[:, :prompt_len] = -100
    
    # Forward
    emotional_model.adapters.train()
    outputs = emotional_model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        labels=labels
    )
    
    loss = outputs.loss
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(emotional_model.adapters.parameters(), 1.0)
    optimizer.step()
    
    return loss.item()

# Train for several epochs
NUM_EPOCHS = 5

print("Training emotion adapters...")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    
    # Shuffle data
    import random
    shuffled = random.sample(TRAINING_DATA, len(TRAINING_DATA))
    
    for prompt, emotion_name, target in tqdm(shuffled, desc=f"Epoch {epoch+1}"):
        loss = train_step(prompt, emotion_name, target)
        total_loss += loss
    
    avg_loss = total_loss / len(TRAINING_DATA)
    print(f"Epoch {epoch+1}: avg_loss = {avg_loss:.4f}")

emotional_model.adapters.eval()
print("\nTraining complete!")

## 7. Test Trained Model

Let's see if training improved emotional differentiation.

In [ ]:
# Test on training-like prompts
test_prompts = [
    "What do you think about this approach?",
    "The build keeps failing.",
    "We finally got it working!",
    "This is definitely the right way.",
]

for prompt in test_prompts:
    print("=" * 60)
    print(f"PROMPT: {prompt}")
    print("-" * 60)
    
    for name in ["curious", "frustrated", "excited", "confident", "skeptical"]:
        emotion = EMOTIONS[name]
        response = generate_with_emotion(prompt, emotion, max_tokens=40)
        print(f"\n[{name}]: {response[:100]}..." if len(response) > 100 else f"\n[{name}]: {response}")
    print()

## 8. Visualize Emotion Modulation

Let's see how different emotions change the adapter outputs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_emotion_modulation():
    """Visualize how emotions modulate the adapters."""
    fig, axes = plt.subplots(2, len(emotional_model.adapter_indices), figsize=(15, 6))
    
    emotion_names = list(EMOTIONS.keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(emotion_names)))
    
    for col, (layer_idx, adapter) in enumerate(emotional_model.adapters.items()):
        scales = []
        biases = []
        
        for name in emotion_names:
            emotion = EMOTIONS[name].to_tensor(device)
            scale, bias = adapter.get_modulation(emotion)
            scales.append(scale.cpu().numpy())
            biases.append(bias.cpu().numpy())
        
        # Plot scale distributions
        ax = axes[0, col]
        for i, (name, scale) in enumerate(zip(emotion_names, scales)):
            ax.hist(scale, bins=50, alpha=0.5, label=name, color=colors[i])
        ax.set_title(f"Layer {layer_idx} - Scale")
        ax.set_xlabel("Scale value")
        if col == 0:
            ax.set_ylabel("Count")
        
        # Plot bias distributions
        ax = axes[1, col]
        for i, (name, bias) in enumerate(zip(emotion_names, biases)):
            ax.hist(bias, bins=50, alpha=0.5, label=name, color=colors[i])
        ax.set_title(f"Layer {layer_idx} - Bias")
        ax.set_xlabel("Bias value")
        if col == 0:
            ax.set_ylabel("Count")
    
    # Add legend
    axes[0, -1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig("emotion_modulation.png", dpi=150, bbox_inches='tight')
    plt.show()

visualize_emotion_modulation()
print("Visualization saved to emotion_modulation.png")

In [ ]:
def compare_emotion_stats():
    """Compare statistics of modulation across emotions."""
    stats = {}
    
    for name in EMOTIONS:
        emotion = EMOTIONS[name].to_tensor(device)
        layer_stats = []
        
        for layer_idx, adapter in emotional_model.adapters.items():
            scale, bias = adapter.get_modulation(emotion)
            layer_stats.append({
                "scale_mean": scale.mean().item(),
                "scale_std": scale.std().item(),
                "bias_mean": bias.mean().item(),
                "bias_std": bias.std().item(),
            })
        
        stats[name] = layer_stats
    
    # Print comparison
    print("Emotion Modulation Statistics (averaged across layers)")
    print("=" * 60)
    print(f"{'Emotion':<12} {'Scale Mean':>12} {'Scale Std':>12} {'Bias Mean':>12} {'Bias Std':>12}")
    print("-" * 60)
    
    for name, layer_stats in stats.items():
        avg_scale_mean = np.mean([s["scale_mean"] for s in layer_stats])
        avg_scale_std = np.mean([s["scale_std"] for s in layer_stats])
        avg_bias_mean = np.mean([s["bias_mean"] for s in layer_stats])
        avg_bias_std = np.mean([s["bias_std"] for s in layer_stats])
        
        print(f"{name:<12} {avg_scale_mean:>12.4f} {avg_scale_std:>12.4f} {avg_bias_mean:>12.4f} {avg_bias_std:>12.4f}")

compare_emotion_stats()

## 9. Save Trained Adapters

Save the trained emotion adapters for later use.

In [ ]:
import os

os.makedirs("models", exist_ok=True)

torch.save({
    'adapters': emotional_model.adapters.state_dict(),
    'adapter_indices': emotional_model.adapter_indices,
    'hidden_dim': hidden_dim,
    'base_model': MODEL_NAME,
}, 'models/emotion_adapters.pt')

print("Saved to models/emotion_adapters.pt")

## 10. Interactive Demo

Try the emotional model interactively!

In [ ]:
def interactive_chat():
    """Interactive chat with emotional AI."""
    print("=" * 60)
    print("EMOTIONAL AI CHAT")
    print("=" * 60)
    print("\nCommands:")
    print("  /emotion <name>  - Set emotion (curious, frustrated, excited, etc.)")
    print("  /list            - List available emotions")
    print("  /quit            - Exit")
    print("\nCurrent emotion: neutral")
    print("-" * 60)
    
    current_emotion = "neutral"
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nBye!")
            break
        
        if not user_input:
            continue
        
        if user_input.startswith("/"):
            parts = user_input.split()
            cmd = parts[0].lower()
            
            if cmd == "/quit":
                print("Bye!")
                break
            elif cmd == "/list":
                print("Available emotions:")
                for name, e in EMOTIONS.items():
                    print(f"  {name}: {e}")
            elif cmd == "/emotion" and len(parts) > 1:
                name = parts[1].lower()
                if name in EMOTIONS:
                    current_emotion = name
                    print(f"Emotion set to: {name} ({EMOTIONS[name]})")
                else:
                    print(f"Unknown emotion: {name}. Use /list to see options.")
            continue
        
        # Generate response
        emotion = EMOTIONS[current_emotion]
        response = generate_with_emotion(user_input, emotion, max_tokens=80)
        print(f"\nAI [{current_emotion}]: {response}")

# Uncomment to run interactive chat
# interactive_chat()

## Summary

This notebook demonstrates:

1. **Emotion State** - 4D continuous emotion space (valence, arousal, curiosity, confidence)

2. **Emotion Adapters** - Layers that modulate hidden states via: `h' = h * (1 + α·scale) + bias`

3. **Integration** - Adapters injected at multiple depths in the transformer

4. **Training** - Supervised on emotion-response pairs

5. **Generation** - Different emotions produce different outputs for the same prompt

### Key Insight

The model *cannot* produce the same output under different emotional states - the emotions literally change how the network computes. This is different from prompting "act excited" which is just text that might be ignored.

### Next Steps

1. Train emotion predictor with RL (learn when to use which emotions)
2. Fine-tune base model for autonomous personality
3. Scale to larger models (7B+)
4. Integrate all components